In [1]:
import torch
import numpy as np
from torch.autograd.functional import jvp

In [8]:
torch.norm(torch.randn(10000, 3), dim=1).mean()

tensor(1.5920)

In [11]:
torch.norm(torch.tensor([[1.0,1,1]]), dim =1)

tensor([1.7321])

In [6]:
torch.randn(10000, 3).abs().mean()

tensor(0.7964)

In [2]:
def make_orthogonal_batch(a):
    """
    a: Tensor of shape (batch_size, dim)
    Returns: Tensor of shape (batch_size, dim), each vector orthogonal to corresponding input
    """
    # Normalize input vectors (shape: [B, D])
    a_norm = a / a.norm(dim=1, keepdim=True)

    # Random vectors (same shape as input)
    b = torch.randn_like(a)

    # Dot product per batch (shape: [B])
    proj_coeff = torch.sum(a_norm * b, dim=1, keepdim=True)

    # Remove projection of b onto a => b_orth = b - proj_a(b)
    b_orth = b - proj_coeff * a_norm

    return b_orth


In [ ]:
normal
orthorgonal
Jacobian prod, forward
angle comparison

In [ ]:
def conformality_angle_loss(f, z):
    #sample batchsize pairs of orthogonal unit vectors
    bs = len(z)
    u = torch.randn_like(z)
    u = u / (u.norm(dim=1, keepdim=True) + 1e-8)

    v = make_orthogonal_batch(u)
    v = v / (v.norm(dim=1, keepdim=True) + 1e-8)


    Jv = jvp(f, z, v, create_graph=True)[1]
    Ju = jvp(f, z, u, create_graph=True)[1]

    # Compute the angle between the two vectors
    cos_angle = torch.sum(Ju * Jv, dim=1)
    # cos_angle = torch.clamp(cos_angle, -1.0, 1.0)  # Ensure values are in [-1, 1]
    Jv_len = Jv.norm(dim=1)
    Ju_len = Ju.norm(dim=1)

    len_diff = torch.abs(Jv_len - Ju_len)

    print(u,v, Jv, Ju, cos_angle, len_diff)



    return Ju, Jv



In [28]:
def func(x):
    return x*3

In [29]:
u, v = conformality_angle_loss(func, torch.randn(5, 2))  # Example usage
torch.norm(u, dim=1), torch.norm(v, dim=1)  # Check norms of u and v

tensor([[ 0.9809, -0.1947],
        [-0.9199,  0.3921],
        [ 0.9805, -0.1964],
        [ 0.9999, -0.0129],
        [-0.1218,  0.9926]]) tensor([[-0.1947, -0.9809],
        [-0.3921, -0.9199],
        [-0.1964, -0.9805],
        [ 0.0129,  0.9999],
        [-0.9926, -0.1218]]) tensor([[-0.5841, -2.9426],
        [-1.1763, -2.7598],
        [-0.5892, -2.9416],
        [ 0.0387,  2.9997],
        [-2.9777, -0.3654]]) tensor([[ 2.9426, -0.5841],
        [-2.7598,  1.1763],
        [ 2.9416, -0.5892],
        [ 2.9998, -0.0387],
        [-0.3654,  2.9777]]) tensor([ 0.0000e+00, -1.6689e-06,  0.0000e+00, -9.0227e-06, -3.8147e-06]) tensor([4.7684e-07, 0.0000e+00, 0.0000e+00, 5.2452e-06, 2.3842e-07])


(tensor([3.0000, 3.0000, 3.0000, 3.0000, 3.0000]),
 tensor([3.0000, 3.0000, 3.0000, 3.0000, 3.0000]))

In [24]:
proj_coeff = torch.sum(u * v, dim=1, keepdim=True)
print(proj_coeff)

tensor([[ 1.3028e+01],
        [ 1.0729e+02],
        [ 4.2450e+00],
        [-1.1184e+02],
        [ 1.8786e-03]], grad_fn=<SumBackward1>)


In [47]:
torch.dot(u[ind], (torch.tensor([-u[ind][1],u[ind][0]])))  # Check dot product of first pair of vectors

tensor(0.)

In [43]:
def make_orthogonal(a):
    # a: (n,) vector
    a = a / a.norm()  # normalize for stability
    b = torch.randn_like(a)
    # Remove projection of b onto a
    b = b - torch.dot(a, b) * a
    return b

a = torch.randn(2000)
b = make_orthogonal(a)

print("a ⋅ b =", torch.dot(a, b))  # Should be ~0


a ⋅ b = tensor(1.1444e-05)


In [31]:
a = torch.tensor([1.0, 2.0, 3.0])
# Pick any vector not parallel to a
temp = torch.tensor([0.0, 0.0, 1.0])
if torch.allclose(torch.cross(a, temp), torch.tensor([0.0, 0.0, 0.0])):
    temp = torch.tensor([1.0, 0.0, 0.0])  # fallback

b = torch.cross(a, temp)  # Orthogonal to both a and temp

print("a ⋅ b =", torch.dot(a, b))  # Should be 0

a ⋅ b = tensor(0.)


/tmp/ipykernel_8343/2096908151.py:4: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)
  if torch.allclose(torch.cross(a, temp), torch.tensor([0.0, 0.0, 0.0])):


In [44]:
torch.dot(a, -a)  # Should be 0

tensor(-2082.2822)